# # 1) IMPORTS

In [1]:
# %%
import os
import json
import random
import logging
import re
import unicodedata
from collections import Counter
from pathlib import Path
from time import perf_counter
from typing import Any, Dict, List, Tuple

import numpy as np
import soundfile as sf
import librosa
import torch
from tqdm.auto import tqdm

from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline,
)

# # 2) CONFIG

In [2]:
# %%
from pathlib import Path
import os
import random
import logging
import numpy as np
import torch

# ===== Portable project/data paths =====
# Có thể override bằng env nếu cần
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", str(Path.cwd().parent))).resolve()
DATA_ROOT = Path(os.getenv("DATA_ROOT", str(PROJECT_ROOT / "data"))).resolve()

RAW_ROOT = Path(os.getenv("RAW_ROOT", str(DATA_ROOT / "VNEMOS"))).resolve()
PROC_WAV_DIR = Path(os.getenv("PROC_WAV_DIR", str(DATA_ROOT / "wavs16k"))).resolve()
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", str(DATA_ROOT / "transcripts"))).resolve()

# Cache HF mặc định về HOME của máy hiện tại, không dùng /mnt/d
HF_CACHE_DIR = Path(
    os.getenv("HF_HOME", str(Path.home() / ".cache" / "hf_cache"))
).resolve()

# ===== Models =====
# Debug ổn định trước bằng base; khi mọi thứ ổn thì đổi sang large
ASR_MODEL_ID = os.getenv("ASR_MODEL_ID", "vinai/PhoWhisper-base")
PUNC_MODEL_ID = os.getenv("PUNC_MODEL_ID", "vinai/bartpho-word-base")

HF_REVISION = os.getenv("HF_REVISION", "main")
HF_LOCAL_ONLY = os.getenv("HF_LOCAL_ONLY", "0") == "1"

# ===== Audio =====
TARGET_SR = 16000

# ===== Split =====
SPLIT_RATIOS = {
    "train": 0.70,
    "valid": 0.15,
    "test": 0.15,
}
SPLIT_TRIALS = 200
RANDOM_SEED = 42

# ===== Devices =====
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Môi trường mới dễ lỗi FP16/CUBLAS -> mặc định dùng FP32 cho ổn định
ASR_DTYPE = torch.float32

# Để punctuation ở CPU để giảm VRAM
PUNC_DEVICE = "cpu"

# Chỉ tạo các thư mục output/cache có quyền ghi
for p in [PROC_WAV_DIR, OUTPUT_DIR, HF_CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(RANDOM_SEED)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_ROOT    :", DATA_ROOT)
print("RAW_ROOT     :", RAW_ROOT)
print("PROC_WAV_DIR :", PROC_WAV_DIR)
print("OUTPUT_DIR   :", OUTPUT_DIR)
print("HF_CACHE_DIR :", HF_CACHE_DIR)
print("DEVICE       :", DEVICE)
print("ASR_DTYPE    :", ASR_DTYPE)
print("PUNC_DEVICE  :", PUNC_DEVICE)

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
    print("CUDA version :", torch.version.cuda)

if not RAW_ROOT.exists():
    raise FileNotFoundError(f"RAW_ROOT does not exist: {RAW_ROOT}")

PROJECT_ROOT : /home/emotalk/mer2
DATA_ROOT    : /home/emotalk/mer2/data
RAW_ROOT     : /home/emotalk/mer2/data/VNEMOS
PROC_WAV_DIR : /home/emotalk/mer2/data/wavs16k
OUTPUT_DIR   : /home/emotalk/mer2/data/transcripts
HF_CACHE_DIR : /home/emotalk/.cache/hf_cache
DEVICE       : cuda
ASR_DTYPE    : torch.float32
PUNC_DEVICE  : cpu
GPU          : NVIDIA GeForce RTX 4060 Ti
CUDA version : 12.8


In [3]:
# %%
wav_like = sorted(
    p for p in RAW_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() == ".wav"
)

print("RAW_ROOT:", RAW_ROOT.resolve())
print("Total wav files:", len(wav_like))
print("First 10 wav files:")
for p in wav_like[:10]:
    print(" -", p)

RAW_ROOT: /home/emotalk/mer2/data/VNEMOS
Total wav files: 250
First 10 wav files:
 - /home/emotalk/mer2/data/VNEMOS/angry/Copy of Angry_scvmc12-00.16.21.029-00.16.25.280-seg5.wav
 - /home/emotalk/mer2/data/VNEMOS/angry/Copy of Angry_scvmc15-00.09.41.333-00.09.43.733-seg1.wav
 - /home/emotalk/mer2/data/VNEMOS/angry/Copy of Angry_scvmc15-00.09.45.505-00.09.52.323-seg2.wav
 - /home/emotalk/mer2/data/VNEMOS/angry/Copy of Angry_scvmc15-00.33.11.861-00.33.18.082-seg8.wav
 - /home/emotalk/mer2/data/VNEMOS/angry/Copy of Angry_scvmc16-00.02.12.124-00.02.17.518-seg1.wav
 - /home/emotalk/mer2/data/VNEMOS/angry/Copy of Angry_scvmc16-00.28.56.600-00.29.03.334-seg2.wav
 - /home/emotalk/mer2/data/VNEMOS/angry/Copy of Angry_scvmc16-00.33.30.273-00.33.35.664-seg3.wav
 - /home/emotalk/mer2/data/VNEMOS/angry/Copy of Angry_scvmc17-00.24.50.801-00.24.58.204-seg1.wav
 - /home/emotalk/mer2/data/VNEMOS/angry/Copy of Angry_scvmc17-00.25.00.160-00.25.07.845-seg2.wav
 - /home/emotalk/mer2/data/VNEMOS/angry/Copy 

# # 3) TEXT + PATH HELPERS

In [4]:
# %%
SPACE_RE = re.compile(r"\s+")
MULTI_PUNC_RE = re.compile(r"([,.;:!?])\1+")
CLIP_SUFFIX_RE = re.compile(r"(?:[_-](?:clip|seg|segment|part)\d+)$", flags=re.IGNORECASE)
TIME_TRAIL_RE = re.compile(r"(?:[_-]?\d{1,2}[._]\d{2}[._]\d{2})+$", flags=re.IGNORECASE)

def slugify(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text))
    text = text.encode("ascii", "ignore").decode("ascii")
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")

def normalize_whitespace(text: str) -> str:
    return SPACE_RE.sub(" ", str(text or "")).strip()

def normalize_transcript(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text or ""))
    text = text.replace("…", "...")
    text = normalize_whitespace(text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"([,.;:!?])([^\s\"'\)\]\}])", r"\1 \2", text)
    text = MULTI_PUNC_RE.sub(r"\1", text)
    return text.strip()

def infer_source_group(rel_path: Path) -> str:
    stem = slugify(rel_path.stem)
    stem = CLIP_SUFFIX_RE.sub("", stem)
    stem = TIME_TRAIL_RE.sub("", stem)
    return stem or slugify(rel_path.parent.name) or "unknown_source"

def parse_vnemos_path(wav_path: Path, raw_root: Path) -> Dict[str, str]:
    rel = wav_path.relative_to(raw_root)
    parts = rel.parts

    # Cấu trúc hiện tại: emotion/file.wav
    if len(parts) < 2:
        raise ValueError(f"Unexpected VNEMOS path: {wav_path}")

    emotion = slugify(parts[0])

    # Chưa có speaker folder rõ ràng trong cấu trúc hiện tại
    speaker_id = "unknown"

    # Dùng tên file để tạo source_group
    source_group = infer_source_group(rel)

    # Split theo source_group
    group_id = source_group

    utterance_id = slugify("__".join(rel.with_suffix("").parts))

    return {
        "utterance_id": utterance_id,
        "emotion": emotion,
        "speaker_id": speaker_id,
        "source_group": source_group,
        "group_id": group_id,
        "relative_path": str(rel),
    }

# # 4) LOAD MODELS

In [5]:
# %%
from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline,
)
from time import perf_counter

logging.info("Loading ASR model from Hugging Face Hub ...")
t0 = perf_counter()

asr_processor = AutoProcessor.from_pretrained(
    ASR_MODEL_ID,
    revision=HF_REVISION,
    cache_dir=str(HF_CACHE_DIR),
    local_files_only=HF_LOCAL_ONLY,
)

asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    ASR_MODEL_ID,
    revision=HF_REVISION,
    cache_dir=str(HF_CACHE_DIR),
    local_files_only=HF_LOCAL_ONLY,
    torch_dtype=ASR_DTYPE,
    low_cpu_mem_usage=True,
    use_safetensors=False,
).eval()

if DEVICE == "cuda":
    asr_model = asr_model.to("cuda")

asr_pipe = pipeline(
    task="automatic-speech-recognition",
    model=asr_model,
    tokenizer=asr_processor.tokenizer,
    feature_extractor=asr_processor.feature_extractor,
    device=0 if DEVICE == "cuda" else -1,
    torch_dtype=ASR_DTYPE,
)

logging.info(f"ASR ready in {perf_counter() - t0:.1f}s")

logging.info("Loading punctuation model ...")
t0 = perf_counter()

punc_tok = AutoTokenizer.from_pretrained(
    PUNC_MODEL_ID,
    revision=HF_REVISION,
    cache_dir=str(HF_CACHE_DIR),
    local_files_only=HF_LOCAL_ONLY,
    use_fast=True,
)

punc_model = AutoModelForSeq2SeqLM.from_pretrained(
    PUNC_MODEL_ID,
    revision=HF_REVISION,
    cache_dir=str(HF_CACHE_DIR),
    local_files_only=HF_LOCAL_ONLY,
).eval().to(PUNC_DEVICE)

logging.info(f"Punctuation ready in {perf_counter() - t0:.1f}s on {PUNC_DEVICE}")

if torch.cuda.is_available():
    print("GPU allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
    print("GPU reserved  GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))

12:48:09 | INFO | Loading ASR model from Hugging Face Hub ...
12:48:10 | INFO | HTTP Request: HEAD https://huggingface.co/vinai/PhoWhisper-base/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
12:48:10 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
12:48:10 | INFO | HTTP Request: HEAD https://huggingface.co/vinai/PhoWhisper-base/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
12:48:10 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/vinai/PhoWhisper-base/7ebdb9e88f5cc5271fb88f4d642c82ff9388650e/preprocessor_config.json "HTTP/1.1 200 OK"
12:48:10 | INFO | HTTP Request: HEAD https://huggingface.co/vinai/PhoWhisper-base/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
12:48:11 | INFO | HTTP Request: HEAD https://huggingface.co/vinai/PhoWhisper-base/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redire

Loading weights:   0%|          | 0/246 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to proj_out.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
12:48:17 | INFO | HTTP Request: GET https://huggingface.co/api/models/vinai/PhoWhisper-base "HTTP/1.1 200 OK"
12:48:17 | INFO | HTTP Request: HEAD https://huggingface.co/vinai/PhoWhisper-base/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
12:48:17 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/vinai/PhoWhisper-base/7ebdb9e88f5cc5271fb88f4d642c82ff9388650e/generation_config.json "HTTP/1.1 200 OK"
12:48:17 | INFO | HTTP Request: GET https://huggingface.co/api/models/vinai/PhoWhisper-base/commits/main "HTTP/1.1 200 OK"
`torch_dtype` is deprecated! Use `dtype` instead!
12:48:17 | INFO | ASR ready in 7.6s
12:48:17 | INFO | Loading punctuation model ...
12:48:17 | INFO | 

Loading weights:   0%|          | 0/265 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
12:48:21 | INFO | HTTP Request: GET https://huggingface.co/api/models/vinai/bartpho-word-base "HTTP/1.1 200 OK"
12:48:21 | INFO | HTTP Request: GET https://huggingface.co/api/models/vinai/bartpho-word-base/commits/main "HTTP/1.1 200 OK"
12:48:21 | INFO | HTTP Request: HEAD https://huggingface.co/vinai/bartpho-word-base/resolve/main/generation_config.json "HTTP/1.1 404 Not Found"
12:48:21 | INFO | HTTP Request: GET https://huggingface.co/api

GPU allocated GB: 0.371
GPU reserved  GB: 0.389


# # 5) ASR FUNCTIONS

In [6]:
# %%
@torch.inference_mode()
def restore_punctuation(text: str) -> str:
    text = normalize_whitespace(text)
    if not text:
        return ""

    enc = punc_tok(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    )
    enc.pop("token_type_ids", None)
    enc = {k: v.to(PUNC_DEVICE) for k, v in enc.items()}

    out = punc_model.generate(
        **enc,
        max_length=min(enc["input_ids"].shape[1] + 24, 512),
        do_sample=False,
    )

    decoded = punc_tok.decode(out[0], skip_special_tokens=True)
    return normalize_transcript(decoded)

def transcribe_audio(audio_path: str) -> str:
    audio_array, sr = sf.read(audio_path, dtype="float32", always_2d=True)
    audio_array = audio_array.mean(axis=1).astype(np.float32)

    if sr != TARGET_SR:
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=TARGET_SR).astype(np.float32)
        sr = TARGET_SR

    result = asr_pipe(
        {"array": audio_array, "sampling_rate": sr},
        generate_kwargs={
            "language": "vi",
            "task": "transcribe",
        },
    )

    return normalize_whitespace(result.get("text", ""))

# # 6) COLLECT + PREPROCESS AUDIO

In [7]:
# %%
def collect_audio_items(raw_root: Path) -> List[Dict[str, Any]]:
    if not raw_root.exists():
        raise FileNotFoundError(f"RAW_ROOT does not exist: {raw_root.resolve()}")

    wav_files = sorted(
        p for p in raw_root.rglob("*")
        if p.is_file() and p.suffix.lower() == ".wav"
    )

    logging.info(f"Scanning audio under: {raw_root.resolve()}")
    logging.info(f"Matched wav files   : {len(wav_files)}")

    items = []
    for wav_path in wav_files:
        try:
            meta = parse_vnemos_path(wav_path, raw_root)
            items.append({
                "utterance_id": meta["utterance_id"],
                "emotion": meta["emotion"],
                "speaker_id": meta["speaker_id"],
                "source_group": meta["source_group"],
                "group_id": meta["group_id"],
                "relative_path": meta["relative_path"],
                "wav_original": str(wav_path),
            })
        except Exception as e:
            logging.warning(f"Skip malformed path {wav_path}: {e}")

    return items

def preprocess_audio(src_path: str, dst_path: str, target_sr: int = 16000) -> float:
    y, sr = sf.read(src_path, dtype="float32", always_2d=True)
    y = y.mean(axis=1)

    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)

    peak = np.max(np.abs(y))
    if peak > 1.0:
        y = y / (peak + 1e-9)

    sf.write(dst_path, y, target_sr, subtype="PCM_16")
    return len(y) / target_sr

items = collect_audio_items(RAW_ROOT)
logging.info(f"Found {len(items)} wav files")

usable_items = []
failed_preprocess = []

for item in tqdm(items, desc="Preprocess"):
    dst = PROC_WAV_DIR / f"{item['utterance_id']}.wav"

    try:
        if not dst.exists():
            duration = preprocess_audio(item["wav_original"], str(dst), TARGET_SR)
        else:
            info = sf.info(str(dst))
            duration = info.frames / info.samplerate

        item["wav_path"] = str(dst)
        item["sample_rate"] = TARGET_SR
        item["duration"] = float(duration)
        usable_items.append(item)

    except Exception as e:
        failed_preprocess.append({
            "utterance_id": item["utterance_id"],
            "wav_original": item["wav_original"],
            "error": repr(e),
        })

items = usable_items
logging.info(f"Usable audio files: {len(items)}")

if failed_preprocess:
    with open(OUTPUT_DIR / "preprocess_errors.json", "w", encoding="utf-8") as f:
        json.dump(failed_preprocess, f, ensure_ascii=False, indent=2)
    logging.warning(f"Saved preprocess errors: {len(failed_preprocess)}")

12:48:21 | INFO | Scanning audio under: /home/emotalk/mer2/data/VNEMOS
12:48:21 | INFO | Matched wav files   : 250
12:48:21 | INFO | Found 250 wav files


Preprocess:   0%|          | 0/250 [00:00<?, ?it/s]

12:48:21 | INFO | Usable audio files: 250


In [8]:
# %%
def choose_transcript_final(
    raw_asr: str,
    punct: str = "",
    verified: str | None = None,
) -> str:

    verified = normalize_transcript(verified) if verified else ""
    punct = normalize_transcript(punct) if punct else ""
    raw_asr = normalize_transcript(raw_asr) if raw_asr else ""

    if verified:
        return verified
    if punct:
        return punct
    return raw_asr

# # 7) TRANSCRIBE + BUILD RECORDS

In [9]:
# %%
def choose_transcript_final(raw_asr: str, punct: str = "", verified: str | None = None) -> str:
    verified = normalize_transcript(verified) if verified else ""
    punct = normalize_transcript(punct) if punct else ""
    raw_asr = normalize_transcript(raw_asr) if raw_asr else ""

    if verified:
        return verified
    if punct:
        return punct
    return raw_asr

def build_record(item: Dict[str, Any]) -> Dict[str, Any]:
    raw_asr = transcribe_audio(item["wav_path"])
    punct = restore_punctuation(raw_asr) if raw_asr else ""

    transcript_final = choose_transcript_final(
        raw_asr=raw_asr,
        punct=punct,
        verified=None,
    )

    return {
        "utterance_id": item["utterance_id"],
        "emotion": item["emotion"],
        "audio_path": item["wav_path"],
        "duration": round(float(item["duration"]), 4),
        "source_group": item["source_group"],
        "group_id": item["group_id"],
        "transcript_final": transcript_final,
    }

metadata = []
failed_asr = []

debug_items = items[:5]
print("Debug on first", len(debug_items), "files")

for item in tqdm(debug_items, desc="ASR-DEBUG"):
    try:
        metadata.append(build_record(item))
        print(f"[OK] {item['utterance_id']}")
    except Exception as e:
        failed_asr.append({
            "utterance_id": item["utterance_id"],
            "audio_path": item["wav_path"],
            "error": repr(e),
        })
        print(f"[ASR ERROR] {item['utterance_id']} -> {repr(e)}")

print("debug_metadata   =", len(metadata))
print("debug_failed_asr =", len(failed_asr))
if failed_asr:
    print("first_failed     =", failed_asr[0])

if torch.cuda.is_available():
    print("GPU allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
    print("GPU reserved  GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))

Debug on first 5 files


ASR-DEBUG:   0%|          | 0/5 [00:00<?, ?it/s]

A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
12:48:22 | INFO | HTTP Request:

[OK] angry_copy_of_angry_scvmc12_00_16_21_029_00_16_25_280_seg5


12:48:23 | INFO | HTTP Request: HEAD https://huggingface.co/vinai/bartpho-word-base/resolve/refs%2Fpr%2F7/model.safetensors.index.json "HTTP/1.1 404 Not Found"


[OK] angry_copy_of_angry_scvmc15_00_09_41_333_00_09_43_733_seg1


12:48:23 | INFO | HTTP Request: HEAD https://huggingface.co/vinai/bartpho-word-base/resolve/refs%2Fpr%2F7/model.safetensors "HTTP/1.1 302 Found"


[OK] angry_copy_of_angry_scvmc15_00_09_45_505_00_09_52_323_seg2
[OK] angry_copy_of_angry_scvmc15_00_33_11_861_00_33_18_082_seg8
[OK] angry_copy_of_angry_scvmc16_00_02_12_124_00_02_17_518_seg1
debug_metadata   = 5
debug_failed_asr = 0
GPU allocated GB: 0.38
GPU reserved  GB: 0.682


In [10]:
# %%
metadata = []
failed_asr = []

for idx, item in enumerate(tqdm(items, desc="ASR")):
    try:
        metadata.append(build_record(item))
    except Exception as e:
        failed_asr.append({
            "utterance_id": item["utterance_id"],
            "audio_path": item["wav_path"],
            "error": repr(e),
        })
        print(f"[ASR ERROR] {item['utterance_id']} -> {repr(e)}")

    if torch.cuda.is_available() and (idx + 1) % 10 == 0:
        torch.cuda.empty_cache()

logging.info(f"Built records: {len(metadata)}")

if failed_asr:
    with open(OUTPUT_DIR / "asr_errors.json", "w", encoding="utf-8") as f:
        json.dump(failed_asr, f, ensure_ascii=False, indent=2)
    logging.warning(f"Saved ASR errors: {len(failed_asr)}")

ASR:   0%|          | 0/250 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[ASR ERROR] sadness_copy_of_sadness_ctnh_3_00_15_22_045_00_15_52_930_seg2 -> ValueError('You have passed more than 3000 mel input features (> 30 seconds) which automatically enables long-form generation which requires the model to predict timestamp tokens. Please either pass `return_timestamps=True` or make sure to pass no more than 3000 mel input features.')


12:51:54 | INFO | Built records: 249
12:51:54 | WARNING | Saved ASR errors: 1


# # 8) SIMPLE GROUP-AWARE 

In [11]:
# %%
print("num metadata:", len(metadata))
print("num emotions:", Counter(r["emotion"] for r in metadata))
print("num groups  :", len(set(r["group_id"] for r in metadata)))

group_sizes = Counter(r["group_id"] for r in metadata)
largest_groups = sorted(group_sizes.items(), key=lambda x: x[1], reverse=True)[:15]

print("\nLargest groups:")
for gid, n in largest_groups:
    print(f"{gid}: {n}")

num metadata: 249
num emotions: Counter({'angry': 50, 'fear': 50, 'happiness': 50, 'neutral': 50, 'sadness': 49})
num groups  : 249

Largest groups:
copy_of_angry_scvmc12_00_16_21_029_00_16_25_280: 1
copy_of_angry_scvmc15_00_09_41_333_00_09_43_733: 1
copy_of_angry_scvmc15_00_09_45_505_00_09_52_323: 1
copy_of_angry_scvmc15_00_33_11_861_00_33_18_082: 1
copy_of_angry_scvmc16_00_02_12_124_00_02_17_518: 1
copy_of_angry_scvmc16_00_28_56_600_00_29_03_334: 1
copy_of_angry_scvmc16_00_33_30_273_00_33_35_664: 1
copy_of_angry_scvmc17_00_24_50_801_00_24_58_204: 1
copy_of_angry_scvmc17_00_25_00_160_00_25_07_845: 1
copy_of_angry_scvmc17_00_25_13_318_00_25_16_111: 1
copy_of_angry_scvmc20_00_21_28_509_00_21_35_181: 1
copy_of_angry_scvmc20_00_23_43_360_00_23_50_400: 1
copy_of_angry_scvmc20_00_24_07_529_00_24_11_303: 1
copy_of_angry_scvmc20_00_31_55_120_00_31_58_467: 1
copy_of_angry_scvmc20_00_33_21_461_00_33_28_131: 1


In [12]:
# %%
def build_group_table(
    records: List[Dict[str, Any]],
    label_key: str = "emotion",
    group_key: str = "group_id",
) -> List[Dict[str, Any]]:
    table = {}

    for rec in records:
        gid = rec[group_key]
        if gid not in table:
            table[gid] = {
                "group_id": gid,
                "records": [],
                "label_counts": Counter(),
                "n": 0,
            }

        table[gid]["records"].append(rec)
        table[gid]["label_counts"][rec[label_key]] += 1
        table[gid]["n"] += 1

    return list(table.values())


def summarize_split(records: List[Dict[str, Any]], split_name: str) -> Dict[str, Any]:
    counts = Counter(r["emotion"] for r in records)
    return {
        "split": split_name,
        "num_samples": len(records),
        "ratio_realized": round(len(records) / max(1, len(metadata)), 4),
        "num_groups": len(set(r["group_id"] for r in records)),
        "label_counts": dict(sorted(counts.items())),
    }


def subset_stats(groups: List[Dict[str, Any]]) -> Tuple[int, Counter]:
    total_n = sum(g["n"] for g in groups)
    total_counts = Counter()
    for g in groups:
        total_counts.update(g["label_counts"])
    return total_n, total_counts


def subset_score(
    selected_groups: List[Dict[str, Any]],
    all_groups: List[Dict[str, Any]],
    target_ratio: float,
    labels: List[str],
) -> float:
    total_n, total_counts = subset_stats(all_groups)
    sel_n, sel_counts = subset_stats(selected_groups)

    target_n = target_ratio * total_n
    target_label_counts = {
        label: target_ratio * total_counts[label]
        for label in labels
    }

    rem_counts = total_counts - sel_counts
    rem_n = total_n - sel_n

    score = 0.0

    # bám gần target size
    score += 3.0 * abs(sel_n - target_n) / max(1.0, target_n)

    # bám gần target label counts
    for label in labels:
        score += abs(sel_counts[label] - target_label_counts[label]) / max(1.0, target_label_counts[label])

        # selected không được thiếu label nếu có thể
        if total_counts[label] > 0 and sel_counts[label] == 0:
            score += 8.0

        # phần còn lại cũng không nên thiếu label
        if total_counts[label] > 1 and rem_counts[label] == 0:
            score += 8.0

    # phạt nếu overshoot quá mạnh
    if sel_n > target_n * 1.25:
        score += 5.0 * (sel_n - target_n * 1.25) / max(1.0, target_n)

    return score


def choose_group_subset(
    groups: List[Dict[str, Any]],
    target_ratio: float,
    trials: int = 300,
    seed: int = 42,
    tolerance: float = 0.06,
) -> List[Dict[str, Any]]:
    labels = sorted({label for g in groups for label in g["label_counts"]})
    total_n, _ = subset_stats(groups)

    best_subset = None
    best_score = None
    best_ratio_gap = None

    for trial in range(trials):
        rng = random.Random(seed + trial)
        order = groups.copy()
        rng.shuffle(order)
        order.sort(key=lambda g: (g["n"], max(g["label_counts"].values())), reverse=True)

        selected = []

        # greedy add
        for g in order:
            cur_score = subset_score(selected, groups, target_ratio, labels)
            new_score = subset_score(selected + [g], groups, target_ratio, labels)

            sel_n, _ = subset_stats(selected)
            target_n = target_ratio * total_n

            # chấp nhận nếu tốt hơn, hoặc còn quá xa target
            if new_score < cur_score or sel_n < target_n * 0.85:
                selected.append(g)

        # prune remove
        improved = True
        while improved and selected:
            improved = False
            cur_score = subset_score(selected, groups, target_ratio, labels)

            for i in range(len(selected)):
                candidate = selected[:i] + selected[i + 1 :]
                cand_score = subset_score(candidate, groups, target_ratio, labels)
                if cand_score < cur_score:
                    selected = candidate
                    improved = True
                    break

        sel_n, _ = subset_stats(selected)
        ratio_realized = sel_n / max(1, total_n)
        ratio_gap = abs(ratio_realized - target_ratio)
        score = subset_score(selected, groups, target_ratio, labels)

        is_better = False
        if best_subset is None:
            is_better = True
        else:
            # ưu tiên candidate nằm trong tolerance
            best_in_tol = best_ratio_gap <= tolerance
            cur_in_tol = ratio_gap <= tolerance

            if cur_in_tol and not best_in_tol:
                is_better = True
            elif cur_in_tol == best_in_tol:
                if score < best_score:
                    is_better = True

        if is_better:
            best_subset = selected
            best_score = score
            best_ratio_gap = ratio_gap

    return best_subset


def groups_to_records(groups: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out = []
    for g in groups:
        out.extend(g["records"])
    return out


def make_two_stage_group_split(
    records: List[Dict[str, Any]],
    ratios: Dict[str, float],
    label_key: str = "emotion",
    group_key: str = "group_id",
    trials: int = 300,
    seed: int = 42,
) -> Tuple[Dict[str, List[Dict[str, Any]]], Dict[str, Any]]:
    assert abs(ratios["train"] + ratios["valid"] + ratios["test"] - 1.0) < 1e-8

    groups = build_group_table(records, label_key=label_key, group_key=group_key)
    all_group_ids = {g["group_id"] for g in groups}

    # Stage 1: chọn test ~ 15%
    test_groups = choose_group_subset(
        groups=groups,
        target_ratio=ratios["test"],
        trials=trials,
        seed=seed,
        tolerance=0.06,
    )
    test_group_ids = {g["group_id"] for g in test_groups}

    remain_groups = [g for g in groups if g["group_id"] not in test_group_ids]

    # Stage 2: valid ratio trên phần còn lại
    # valid_final = 0.15 toàn tập
    # valid_on_remain = 0.15 / (1 - 0.15)
    valid_ratio_on_remain = ratios["valid"] / (1.0 - ratios["test"])

    valid_groups = choose_group_subset(
        groups=remain_groups,
        target_ratio=valid_ratio_on_remain,
        trials=trials,
        seed=seed + 999,
        tolerance=0.06,
    )
    valid_group_ids = {g["group_id"] for g in valid_groups}

    train_groups = [g for g in remain_groups if g["group_id"] not in valid_group_ids]

    splits = {
        "train": groups_to_records(train_groups),
        "valid": groups_to_records(valid_groups),
        "test": groups_to_records(test_groups),
    }

    for split_name, recs in splits.items():
        for rec in recs:
            rec["split"] = split_name

    summary = {
        split_name: summarize_split(recs, split_name)
        for split_name, recs in splits.items()
    }

    return splits, summary

# # 9) SAVE JSONL + SCHEMA

In [13]:
# %%
splits, split_summary = make_two_stage_group_split(
    records=metadata,
    ratios=SPLIT_RATIOS,
    label_key="emotion",
    group_key="group_id",
    trials=SPLIT_TRIALS,
    seed=RANDOM_SEED,
)

all_with_split = []
for split_name in ["train", "valid", "test"]:
    all_with_split.extend(splits[split_name])

# Save per split
for split_name, recs in splits.items():
    out_path = OUTPUT_DIR / f"{split_name}.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for row in recs:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    logging.info(f"Saved {split_name:5s}: {len(recs):4d} samples -> {out_path}")

# Save full
with open(OUTPUT_DIR / "vnemos_mer_all.jsonl", "w", encoding="utf-8") as f:
    for row in all_with_split:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

split_report = {
    "ratios_target": SPLIT_RATIOS,
    "summary": split_summary,
    "labels": sorted({rec["emotion"] for rec in metadata}),
    "group_key_used": "group_id",
    "group_note": "two-stage group-aware split; exact 70/15/15 may still be impossible if groups are coarse",
    "num_total_samples": len(metadata),
    "num_total_groups": len({rec["group_id"] for rec in metadata}),
}

with open(OUTPUT_DIR / "split_summary.json", "w", encoding="utf-8") as f:
    json.dump(split_report, f, ensure_ascii=False, indent=2)

print(json.dumps(split_report, ensure_ascii=False, indent=2))

12:52:12 | INFO | Saved train:  175 samples -> /home/emotalk/mer2/data/transcripts/train.jsonl
12:52:12 | INFO | Saved valid:   37 samples -> /home/emotalk/mer2/data/transcripts/valid.jsonl
12:52:12 | INFO | Saved test :   37 samples -> /home/emotalk/mer2/data/transcripts/test.jsonl


{
  "ratios_target": {
    "train": 0.7,
    "valid": 0.15,
    "test": 0.15
  },
  "summary": {
    "train": {
      "split": "train",
      "num_samples": 175,
      "ratio_realized": 0.7028,
      "num_groups": 175,
      "label_counts": {
        "angry": 35,
        "fear": 35,
        "happiness": 35,
        "neutral": 35,
        "sadness": 35
      }
    },
    "valid": {
      "split": "valid",
      "num_samples": 37,
      "ratio_realized": 0.1486,
      "num_groups": 37,
      "label_counts": {
        "angry": 8,
        "fear": 8,
        "happiness": 7,
        "neutral": 7,
        "sadness": 7
      }
    },
    "test": {
      "split": "test",
      "num_samples": 37,
      "ratio_realized": 0.1486,
      "num_groups": 37,
      "label_counts": {
        "angry": 7,
        "fear": 7,
        "happiness": 8,
        "neutral": 8,
        "sadness": 7
      }
    }
  },
  "labels": [
    "angry",
    "fear",
    "happiness",
    "neutral",
    "sadness"
  ],
  "group_

# # 10) QUICK PREVIEW

In [14]:
# %%
def print_preview(records: List[Dict[str, Any]], n: int = 3) -> None:
    for row in records[:n]:
        print("=" * 100)
        print("utterance_id    :", row["utterance_id"])
        print("emotion         :", row["emotion"])
        print("audio_path      :", row["audio_path"])
        print("duration        :", row["duration"])
        print("transcript_final:", row["transcript_final"])
        print("split           :", row["split"])

print_preview(all_mer_records, n=5)

NameError: name 'all_mer_records' is not defined